# Hybrid Document Classifier: Encoder Embeddings + SVM

**Status**: 🚧 DRAFT - Concept documented, implementation pending

## The "Hack": Why This Is Interesting

**Problem with pure generation:**
- Decoder can hallucinate on bad scans
- Slow inference (autoregressive token-by-token)
- No interpretable confidence measure

**Our hybrid approach:**
- Use encoder as feature extractor (fast, one forward pass)
- Train SVM on embeddings (interpretable decision boundary)
- Distance to hyperplane = confidence score (auditable!)

## Architecture

```
┌─────────────────────────────────────────────────────────────────┐
│  HYBRID PIPELINE                                                │
│                                                                 │
│  Document Image                                                 │
│       ↓                                                         │
│  [TrOCR Encoder] → 768-dim Embedding                           │
│       ↓                                                         │
│  [SVM Classifier]                                               │
│       ↓                                                         │
│  ┌─────────────────────────────────────────────┐               │
│  │ Prediction: "Invoice" | "Contract" | ...    │               │
│  │ Confidence: Distance to hyperplane          │               │
│  └─────────────────────────────────────────────┘               │
│       ↓                                                         │
│  ┌─────────────┐    ┌──────────────────┐                       │
│  │ High Conf   │    │ Low Confidence   │                       │
│  │ → Auto-route│    │ → Human Review   │                       │
│  └─────────────┘    └──────────────────┘                       │
└─────────────────────────────────────────────────────────────────┘
```

## Why SVM?

1. **Max-margin classifier** - Optimizes for separation, not just accuracy
2. **Works well in high dimensions** - 768-dim embedding space is perfect
3. **Interpretable confidence** - Distance to hyperplane is meaningful
4. **Small data friendly** - Works with 50-100 samples per class
5. **Fast inference** - Single dot product after embedding

## Planned Sections

1. **Load Embeddings** - From notebook 02 or extract on-the-fly
2. **Train SVM** - sklearn.svm.SVC with RBF kernel
3. **Confidence Calibration** - Map distance to probability
4. **Compare Approaches**:
   - SVM confidence vs Decoder token probability
   - Speed comparison (embedding + SVM vs full generation)
5. **Production Pattern** - Cascade: classify → route → OCR if needed

## Key Code Snippet (Preview)

```python
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

# Assume embeddings from notebook 02
# X: (n_samples, 768) - encoder embeddings
# y: (n_samples,) - document labels

# Train SVM
clf = Pipeline([
    ('scaler', StandardScaler()),
    ('svm', SVC(kernel='rbf', probability=True))
])
clf.fit(X_train, y_train)

# Predict with confidence
def classify_document(image):
    embedding = extract_embedding(image)  # From notebook 02
    
    prediction = clf.predict([embedding])[0]
    probabilities = clf.predict_proba([embedding])[0]
    confidence = max(probabilities)
    
    # Decision function = distance to hyperplane (more interpretable)
    decision = clf.decision_function([embedding])
    
    return {
        'class': prediction,
        'confidence': confidence,
        'decision_margin': decision,
        'needs_review': confidence < 0.8
    }
```

## Validation Strategy

The key insight: **We can validate the classifier!**

| Scenario | SVM Confidence | Action |
|----------|----------------|--------|
| High margin, correct class | > 0.9 | Auto-process |
| Low margin, any class | < 0.7 | Human review |
| High margin, wrong class | > 0.9 | Retrain signal! |

This makes the system **auditable** - we know when it's uncertain.

## Document Types (Target Classes)

Based on medical/CRM use case:
- Invoice / Rechnung
- Doctor Letter / Arztbrief  
- Delivery Note / Lieferschein
- Bank Statement / Kontoauszug
- Contract / Vertrag
- Email / Correspondence
- Unknown (→ always human review)

## References

- [SVM Tutorial](https://scikit-learn.org/stable/modules/svm.html)
- [Calibration of Classifiers](https://scikit-learn.org/stable/modules/calibration.html)
- LinkedIn post concept: "Beyond Text Generation: Using OCR Encoders as Feature Extractors"

In [ ]:
# TODO: Implement after notebooks 01 and 02 are stable
# See markdown cell above for the plan
print("🚧 This notebook is a placeholder - implementation coming soon!")